In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_54_try_1.pickle

In [ ]:
%%PandasProfile
### cell 55 ###

# Define question text mappings
question_of_interest_cell67 = 'Which of the following cloud computing platforms do you use? (Select all that apply)'
question_of_interest_old_cell67 = 'Which of the following cloud computing platforms do you use on a regular basis?'
question_of_interest_old_2 = 'Which of the following cloud computing services have you used at work or school in the last 5 years?'

# Standardize column names in 2018 DataFrame
mapping_2018 = {
    'Amazon Web Services (AWS)': 'Amazon Web Services (AWS) ',
    'Google Cloud Platform (GCP)': 'Google Cloud Platform (GCP) ',
    'Microsoft Azure': 'Microsoft Azure ',
    question_of_interest_old_2: question_of_interest_cell67
}
responses_df_2018_cell10.rename(columns=mapping_2018, inplace=True)

# Standardize question text columns in other DataFrames
for df in (responses_df_2021, responses_df_2020, responses_df_2019_cell10):
    df.rename(columns={question_of_interest_old_cell67: question_of_interest_cell67}, inplace=True)

# Extract and clean subset of columns for a given question
def grab_subset_of_data_67(df, question_of_interest):
    cols = df.columns[df.columns.str.contains(question_of_interest, regex=False)]
    mapper = {col: col.rsplit('-', 1)[-1].lstrip() for col in cols}
    return df.loc[:, cols].rename(columns=mapper).dropna(how='all')

# Convert raw counts to percentages per year
def convert_df_of_counts_to_percentages_67(df, df_counts):
    totals = df['year'].value_counts().sort_index()
    df_perc = df_counts.set_index('year').div(totals, axis=0) * 100
    return df_perc.reset_index()

# Combine multiple years, tag with year, and count responses
def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(
        question_of_interest, include_2017=False):
    resp_map = {
        '2017': responses_df_2017,
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10
    }
    years = ['2017','2018','2019','2020','2021','2022'] if include_2017 else ['2018','2019','2020','2021','2022']
    dfs = [grab_subset_of_data_67(resp_map[y], question_of_interest).assign(year=y) for y in years]
    df_combined = pd.concat(dfs, ignore_index=True)
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts

# Execute the pipeline
cloud_df_combined, cloud_df_combined_counts = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(
        question_of_interest_cell67)
cloud_df_combined_percentages = \
    convert_df_of_counts_to_percentages_67(cloud_df_combined, cloud_df_combined_counts)
cloud_df_combined_percentages_cell67 = cloud_df_combined_percentages.loc[:, [
    'year',
    'Amazon Web Services (AWS) ',
    'Google Cloud Platform (GCP) ',
    'Microsoft Azure '
]]
# Melt for plotting and rename the "variable" column to an empty string
df_cell67 = cloud_df_combined_percentages_cell67.melt(
    id_vars=['year'],
    value_vars=['Amazon Web Services (AWS) ', 'Google Cloud Platform (GCP) ', 'Microsoft Azure ']
).rename(columns={'variable': ''})
df_cell67.info()